# Reference analysis notebook

This notebook is provided as reference analysis code for the accepted paper figures. It is not intended to be a standalone reproduction package. Model weights, LoRA adapters, datasets, and intermediate hidden-state files are not included. Local paths under `data/`, `adapters/`, and `outputs/` should be adjusted to the user's environment.


In [ ]:
# Figure: CKA difference heatmap between baseline and ours.
# Required inputs: two square CKA matrices saved as .npy files.
# The notebook loads both matrices, plots ours - baseline, and saves the heatmap plus the difference matrix.

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASELINE_CKA_PATH = Path("data/cka_baseline.npy")
OURS_CKA_PATH = Path("data/cka_ours.npy")
OUTPUT_DIR = Path("outputs/cka_difference")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_cka_matrix(path):
    matrix = np.load(path)
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError(f"Expected a square 2D matrix at {path}, got shape {matrix.shape}")
    return matrix.astype(np.float64)

def plot_cka_difference(base_matrix, ours_matrix, output_path, limit=0.005):
    if limit <= 0:
        raise ValueError("limit must be positive")
    if base_matrix.shape != ours_matrix.shape:
        raise ValueError(f"Matrix shapes must match: {base_matrix.shape} vs {ours_matrix.shape}")
    diff = ours_matrix - base_matrix
    plt.figure(figsize=(10, 8))
    sns.set_theme(style="white")
    ax = sns.heatmap(diff, cmap="RdBu_r", center=0, vmin=-limit, vmax=limit, square=True, cbar_kws={"label": "CKA Difference (Ours - Baseline)"})
    ax.invert_yaxis()
    ticks = np.arange(0, diff.shape[0], 5)
    ax.set_xticks(ticks + 0.5)
    ax.set_yticks(ticks + 0.5)
    ax.set_xticklabels(ticks, rotation=0, fontsize=12)
    ax.set_yticklabels(ticks, rotation=0, fontsize=12)
    plt.title("CKA Structural Difference")
    plt.xlabel("Layer")
    plt.ylabel("Layer")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return diff

In [ ]:
cka_baseline = load_cka_matrix(BASELINE_CKA_PATH)
cka_ours = load_cka_matrix(OURS_CKA_PATH)
cka_difference = plot_cka_difference(cka_baseline, cka_ours, OUTPUT_DIR / "cka_difference_heatmap.png")
np.save(OUTPUT_DIR / "cka_difference.npy", cka_difference)